In [1]:
import os
import pandas as pd
import numpy as np
import cv2
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt

In [2]:
metadata_path = "dataset/HAM10000_metadata.csv"
df = pd.read_csv(metadata_path)
df.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear


In [3]:
image_paths = []
image_folders = [
    "dataset/ham10000_images_part_1",
    "dataset/ham10000_images_part_2"
]
for image_id in df["image_id"]:
    image_path = None
    for folder in image_folders:
        path = os.path.join(folder,image_id + ".jpg")
        if os.path.exists(path):
            image_path = path
            break
    image_paths.append(image_path)
df["image_path"] = image_paths

In [4]:
df["image_path"].isnull().sum()

np.int64(0)

In [5]:
encoder = LabelEncoder()
df["label"] = encoder.fit_transform(df["dx"])

In [6]:
class_mapping = dict(zip(encoder.classes_,encoder.transform(encoder.classes_)))
class_mapping

{'akiec': np.int64(0),
 'bcc': np.int64(1),
 'bkl': np.int64(2),
 'df': np.int64(3),
 'mel': np.int64(4),
 'nv': np.int64(5),
 'vasc': np.int64(6)}

In [7]:
train_data, temp_data = train_test_split(df,test_size=0.30,random_state=42,stratify=df["label"])

In [8]:
val_data, test_data = train_test_split(temp_data,test_size=0.50,random_state=42,stratify=temp_data["label"])

In [9]:
print(train_data.shape,val_data.shape,test_data.shape)

(7010, 9) (1502, 9) (1503, 9)


In [10]:
train_data.to_csv("dataset/train_metadata.csv",index=False)
val_data.to_csv("dataset/val_metadata.csv",index=False)
test_data.to_csv("dataset/test_metadata.csv",index=False)

In [11]:
IMAGE_SIZE = 224
def preprocess_image(image_path):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
    image = cv2.resize(image,(IMAGE_SIZE, IMAGE_SIZE))
    image = image / 255.0
    return image.astype(np.float32)

In [12]:
sample_image = preprocess_image(
    train_data.iloc[0]["image_path"]
)


sample_image.shape

(224, 224, 3)

In [13]:
def create_dataset(data):
    image_paths = data["image_path"].values
    labels = data["label"].values
    dataset = tf.data.Dataset.from_tensor_slices((image_paths,labels))
    def load_image(path,label):
        image = tf.numpy_function(preprocess_image,[path],tf.float32)
        image.set_shape((224,224,3))
        return image,label
    dataset = dataset.map(load_image,num_parallel_calls=tf.data.AUTOTUNE)
    return dataset

In [14]:
train_dataset = create_dataset(train_data)
val_dataset = create_dataset(val_data)
test_dataset = create_dataset(test_data)

In [15]:
data_augmentation = tf.keras.Sequential([

    tf.keras.layers.RandomFlip("horizontal"),

    tf.keras.layers.RandomRotation(0.1),

    tf.keras.layers.RandomZoom(0.1),

    tf.keras.layers.RandomContrast(0.1)

])

In [16]:
def apply_augmentation(image,label):
    image = data_augmentation(image)
    return image,label
train_dataset = train_dataset.map(apply_augmentation,num_parallel_calls=tf.data.AUTOTUNE)

In [17]:
BATCH_SIZE = 32
train_dataset = (train_dataset.shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
val_dataset = (val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
test_dataset = (test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

In [18]:
class_weights = compute_class_weight(class_weight="balanced",classes=np.unique(train_data["label"]),y=train_data["label"])
class_weights

array([ 4.37305053,  2.78174603,  1.30224782, 12.3633157 ,  1.2855309 ,
        0.21338772, 10.11544012])

In [19]:
class_weights = dict(enumerate(class_weights))
class_weights

{0: np.float64(4.37305053025577),
 1: np.float64(2.7817460317460316),
 2: np.float64(1.3022478172023035),
 3: np.float64(12.36331569664903),
 4: np.float64(1.285530900421786),
 5: np.float64(0.21338772031292808),
 6: np.float64(10.115440115440116)}

In [20]:
from tensorflow.keras import layers, models


cnn_batchnorm = models.Sequential([
    layers.Input(shape=(224,224,3)),
    # Block 1
    layers.Conv2D(32,(3,3),padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Conv2D(32,(3,3),padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2,2)),
    # Block 2
    layers.Conv2D(64,(3,3),padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Conv2D(64,(3,3),padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2,2)),
    # Block 3
    layers.Conv2D(128,(3,3),padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2,2)),
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation="relu"),
    layers.Dense(7,activation="softmax")

])


cnn_batchnorm.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 224, 224, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 112, 112, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 158,119 (617.65 KB)

 Trainable params: 157,479 (615.15 KB)

 Non-trainable params: 640 (2.50 KB)

In [21]:
cnn_batchnorm.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"]

)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)

In [22]:
history_basic = cnn_batchnorm.fit(train_dataset,validation_data=val_dataset,epochs=10,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/10


/Users/aximsoft/Documents/untitled folder/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 415s 2s/step - accuracy: 0.3699 - loss: 1.7239 - val_accuracy: 0.0113 - val_loss: 2.2880
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 420s 2s/step - accuracy: 0.4690 - loss: 1.5284 - val_accuracy: 0.0639 - val_loss: 2.3041
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 409s 2s/step - accuracy: 0.4856 - loss: 1.4503 - val_accuracy: 0.4621 - val_loss: 1.4958
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 406s 2s/step - accuracy: 0.4904 - loss: 1.3863 - val_accuracy: 0.4055 - val_loss: 1.6282
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 404s 2s/step - accuracy: 0.4936 - loss: 1.3529 - val_accuracy: 0.4061 - val_loss: 1.9305
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 402s 2s/step - accuracy: 0.5247 - loss: 1.3156 - val_accuracy: 0.4594 - val_loss: 1.5318
Epoch 6: early stopping
Restoring model weights from the end of the best epoch: 3.


In [23]:
train_loss, train_accuracy = cnn_batchnorm.evaluate(train_dataset)
val_loss, val_accuracy = cnn_batchnorm.evaluate(val_dataset)
test_loss, test_accuracy = cnn_batchnorm.evaluate(test_dataset)


220/220 ━━━━━━━━━━━━━━━━━━━━ 89s 396ms/step - accuracy: 0.4829 - loss: 1.4132
47/47 ━━━━━━━━━━━━━━━━━━━━ 18s 377ms/step - accuracy: 0.4621 - loss: 1.4958
47/47 ━━━━━━━━━━━━━━━━━━━━ 18s 377ms/step - accuracy: 0.4291 - loss: 1.5383


In [24]:
cnn_batchnorm.save("models/cnn_batchnorm.keras")

In [21]:
from tensorflow.keras import layers, models


cnn_batchnorm_sgd= models.Sequential([
    layers.Input(shape=(224,224,3)),
    # Block 1
    layers.Conv2D(32,(3,3),padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Conv2D(32,(3,3),padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2,2)),
    # Block 2
    layers.Conv2D(64,(3,3),padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Conv2D(64,(3,3),padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2,2)),
    # Block 3
    layers.Conv2D(128,(3,3),padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2,2)),
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation="relu"),
    layers.Dense(7,activation="softmax")

])


cnn_batchnorm_sgd.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 224, 224, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 112, 112, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 158,119 (617.65 KB)

 Trainable params: 157,479 (615.15 KB)

 Non-trainable params: 640 (2.50 KB)

In [23]:
cnn_batchnorm_sgd.compile(

    optimizer=tf.keras.optimizers.SGD(
        learning_rate=0.0001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)

history_basic__sgd = cnn_batchnorm_sgd.fit(

    train_dataset,

    validation_data=val_dataset,

    epochs=7,

    class_weight=class_weights
)

Epoch 1/7
220/220 ━━━━━━━━━━━━━━━━━━━━ 398s 2s/step - accuracy: 0.0738 - loss: 2.0636 - val_accuracy: 0.0333 - val_loss: 2.1121
Epoch 2/7
220/220 ━━━━━━━━━━━━━━━━━━━━ 404s 2s/step - accuracy: 0.0916 - loss: 1.9747 - val_accuracy: 0.0859 - val_loss: 2.1304
Epoch 3/7
220/220 ━━━━━━━━━━━━━━━━━━━━ 403s 2s/step - accuracy: 0.1327 - loss: 1.9123 - val_accuracy: 0.1578 - val_loss: 2.0507
Epoch 4/7
220/220 ━━━━━━━━━━━━━━━━━━━━ 404s 2s/step - accuracy: 0.1839 - loss: 1.8698 - val_accuracy: 0.2344 - val_loss: 1.9912
Epoch 5/7
220/220 ━━━━━━━━━━━━━━━━━━━━ 405s 2s/step - accuracy: 0.2418 - loss: 1.8384 - val_accuracy: 0.3043 - val_loss: 1.9295
Epoch 6/7
220/220 ━━━━━━━━━━━━━━━━━━━━ 408s 2s/step - accuracy: 0.2903 - loss: 1.8125 - val_accuracy: 0.3622 - val_loss: 1.8523
Epoch 7/7
220/220 ━━━━━━━━━━━━━━━━━━━━ 401s 2s/step - accuracy: 0.3387 - loss: 1.7894 - val_accuracy: 0.3735 - val_loss: 1.8306


In [24]:
train_loss, train_accuracy = cnn_batchnorm_sgd.evaluate(train_dataset)
val_loss, val_accuracy = cnn_batchnorm_sgd.evaluate(val_dataset)
test_loss, test_accuracy = cnn_batchnorm_sgd.evaluate(test_dataset)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

220/220 ━━━━━━━━━━━━━━━━━━━━ 90s 399ms/step - accuracy: 0.3331 - loss: 1.8523
47/47 ━━━━━━━━━━━━━━━━━━━━ 18s 386ms/step - accuracy: 0.3735 - loss: 1.8306
47/47 ━━━━━━━━━━━━━━━━━━━━ 18s 387ms/step - accuracy: 0.3253 - loss: 1.8634
Test Loss: 1.8634098768234253
Test Accuracy: 0.3253493010997772


In [25]:
cnn_batchnorm_sgd.save("models/cnn_batchnorm_sgd.keras")

In [26]:
from tensorflow.keras import layers, models


cnn_batchnorm_RMSprop= models.Sequential([
    layers.Input(shape=(224,224,3)),
    # Block 1
    layers.Conv2D(32,(3,3),padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Conv2D(32,(3,3),padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2,2)),
    # Block 2
    layers.Conv2D(64,(3,3),padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Conv2D(64,(3,3),padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2,2)),
    # Block 3
    layers.Conv2D(128,(3,3),padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2,2)),
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation="relu"),
    layers.Dense(7,activation="softmax")

])


cnn_batchnorm_RMSprop.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_5 (Conv2D)               │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 224, 224, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_6 (Activation)       │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_7 (Activation)       │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 112, 112, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_8 (Activation)       │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_9 (Activation)       │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 158,119 (617.65 KB)

 Trainable params: 157,479 (615.15 KB)

 Non-trainable params: 640 (2.50 KB)

In [27]:
cnn_batchnorm_RMSprop.compile(

    optimizer=tf.keras.optimizers.RMSprop(
        learning_rate=0.0001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)

history_cnn_batchnorm_RMSprop= cnn_batchnorm_RMSprop.fit(
    train_dataset,
    validation_data=val_dataset,

    epochs=5,

    class_weight=class_weights
)

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 404s 2s/step - accuracy: 0.4029 - loss: 1.7178 - val_accuracy: 0.0253 - val_loss: 2.1760
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 407s 2s/step - accuracy: 0.4685 - loss: 1.5291 - val_accuracy: 0.0919 - val_loss: 2.2769
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 405s 2s/step - accuracy: 0.4866 - loss: 1.4408 - val_accuracy: 0.4228 - val_loss: 1.7800
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 407s 2s/step - accuracy: 0.4979 - loss: 1.3834 - val_accuracy: 0.4967 - val_loss: 1.4708
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 406s 2s/step - accuracy: 0.5127 - loss: 1.3536 - val_accuracy: 0.4521 - val_loss: 1.4816


In [28]:
train_loss, train_accuracy = cnn_batchnorm_RMSprop.evaluate(train_dataset)
val_loss, val_accuracy = cnn_batchnorm_RMSprop.evaluate(val_dataset)
test_loss, test_accuracy = cnn_batchnorm_RMSprop.evaluate(test_dataset)


220/220 ━━━━━━━━━━━━━━━━━━━━ 88s 394ms/step - accuracy: 0.5077 - loss: 1.2888
47/47 ━━━━━━━━━━━━━━━━━━━━ 18s 380ms/step - accuracy: 0.4521 - loss: 1.4816
47/47 ━━━━━━━━━━━━━━━━━━━━ 18s 381ms/step - accuracy: 0.4265 - loss: 1.4812


In [29]:
cnn_batchnorm_RMSprop.save("models/cnn_batchnorm_RMSprop.keras")

## Batchsize(64)


In [20]:
train_dataset = create_dataset(train_data)
val_dataset = create_dataset(val_data)
test_dataset = create_dataset(test_data)
 
train_dataset = train_dataset.map(
    apply_augmentation,
    num_parallel_calls=tf.data.AUTOTUNE
)
 
BATCH_SIZE = 64
 
train_dataset = (
    train_dataset
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
 
val_dataset = (
    val_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
 
test_dataset = (
    test_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [21]:
from tensorflow.keras import layers, models


cnn_batchnorm_64= models.Sequential([
    layers.Input(shape=(224,224,3)),
    # Block 1
    layers.Conv2D(32,(3,3),padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Conv2D(32,(3,3),padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2,2)),
    # Block 2
    layers.Conv2D(64,(3,3),padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Conv2D(64,(3,3),padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2,2)),
    # Block 3
    layers.Conv2D(128,(3,3),padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2,2)),
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation="relu"),
    layers.Dense(7,activation="softmax")

])


cnn_batchnorm_64.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 224, 224, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 112, 112, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 158,119 (617.65 KB)

 Trainable params: 157,479 (615.15 KB)

 Non-trainable params: 640 (2.50 KB)

In [26]:
cnn_batchnorm_64.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)
history_basic = cnn_batchnorm_64.fit(train_dataset,validation_data=val_dataset,epochs=8,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 412s 4s/step - accuracy: 0.4689 - loss: 1.4691 - val_accuracy: 0.0446 - val_loss: 2.8440
Epoch 2/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 414s 4s/step - accuracy: 0.4581 - loss: 1.4397 - val_accuracy: 0.0200 - val_loss: 2.5496
Epoch 3/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 412s 4s/step - accuracy: 0.4884 - loss: 1.3711 - val_accuracy: 0.1285 - val_loss: 2.2974
Epoch 4/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 412s 4s/step - accuracy: 0.4879 - loss: 1.3595 - val_accuracy: 0.4953 - val_loss: 1.4946
Epoch 5/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 409s 4s/step - accuracy: 0.4932 - loss: 1.3200 - val_accuracy: 0.3822 - val_loss: 1.6429
Epoch 6/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 419s 4s/step - accuracy: 0.4843 - loss: 1.3043 - val_accuracy: 0.4028 - val_loss: 1.5234
Epoch 7/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 416s 4s/step - accuracy: 0.5137 - loss: 1.2994 - val_accuracy: 0.3442 - val_loss: 1.6799
Epoch 7: early stopping
Restoring model weights from the end of the best epoch: 4.


In [27]:
train_loss, train_accuracy = cnn_batchnorm_64.evaluate(train_dataset)
val_loss, val_accuracy = cnn_batchnorm_64.evaluate(val_dataset)
test_loss, test_accuracy = cnn_batchnorm_64.evaluate(test_dataset)


110/110 ━━━━━━━━━━━━━━━━━━━━ 89s 790ms/step - accuracy: 0.5000 - loss: 1.4273
24/24 ━━━━━━━━━━━━━━━━━━━━ 18s 736ms/step - accuracy: 0.4953 - loss: 1.4946
24/24 ━━━━━━━━━━━━━━━━━━━━ 18s 735ms/step - accuracy: 0.4731 - loss: 1.4977


In [28]:
cnn_batchnorm_64.save("models/cnn_batchnorm_64.keras")

In [31]:
!pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [keras-tuner]


### hyperparameter

In [32]:
import keras_tuner as kt
import tensorflow as tf
from tensorflow.keras import models, layers
from tensorflow.keras.optimizers import Adam, SGD, RMSprop

num_classes = 7

def build_model(hp):

    cnn_batchnorm_64 = models.Sequential([

        layers.Input(shape=(224,224,3)),

        layers.Conv2D(

            filters=hp.Choice(
                "filters_1",
                [32,64]
            ),

            kernel_size=(3,3),

            activation="relu"
        ),

        layers.MaxPooling2D((2,2)),


        layers.Conv2D(

            filters=hp.Choice(
                "filters_2",
                [64,128]
            ),

            kernel_size=(3,3),

            activation="relu"
        ),

        layers.MaxPooling2D((2,2)),


        layers.Flatten(),


        layers.Dense(

            units=hp.Choice(
                "dense_units",
                [64,128]
            ),

            activation="relu"
        ),


        layers.Dense(

            num_classes,

            activation="softmax"
        )

    ])


    learning_rate = hp.Choice(

        "learning_rate",

        [1e-2,1e-3,1e-4]
    )


    optimizer = hp.Choice(

        "optimizer",

        ["adam","rmsprop"]
    )


    if optimizer == "adam":

        opt = Adam(
            learning_rate=learning_rate
        )

    else:

        opt = RMSprop(
            learning_rate=learning_rate
        )


    cnn_batchnorm_64.compile(

        optimizer=opt,

        loss="sparse_categorical_crossentropy",

        metrics=["accuracy"]
    )

    return cnn_batchnorm_64

In [33]:
tuner = kt.RandomSearch(
    build_model,
    objective="val_accuracy",
    max_trials=3,
    overwrite=True,
    directory="tuner",
)

In [37]:
import tensorboard
print(tensorboard.__version__)

2.21.0


In [38]:
tuner.search(
    train_dataset,
    validation_data=val_dataset,
    epochs=5,
    class_weight=class_weights
)

Trial 3 Complete [00h 11m 58s]
val_accuracy: 0.10985352843999863

Best val_accuracy So Far: 0.10985352843999863
Total elapsed time: 00h 11m 58s


In [39]:
best_bn_cnn = tuner.get_best_models(1)[0]

/Users/aximsoft/Documents/untitled folder/.venv/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(store)


In [40]:
best_hps = tuner.get_best_hyperparameters(
    num_trials=1
)[0]
print(best_hps.values)

{'filters_1': 32, 'filters_2': 128, 'dense_units': 64, 'learning_rate': 0.01, 'optimizer': 'adam'}


In [41]:
from tensorflow.keras import layers, models


final_batch= models.Sequential([
    layers.Input(shape=(224,224,3)),
    # Block 1
    layers.Conv2D(32,(3,3),padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Conv2D(32,(3,3),padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2,2)),
    # Block 2
    layers.Conv2D(128,(3,3),padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2,2)),
    layers.GlobalAveragePooling2D(),
    layers.Dense(64,activation="relu"),
    layers.Dense(7,activation="softmax")

])


final_batch.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 224, 224, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 112, 112, 128)  │        36,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 112, 112, 128)  │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 112, 112, 128)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │           455 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 56,615 (221.15 KB)

 Trainable params: 56,231 (219.65 KB)

 Non-trainable params: 384 (1.50 KB)

In [42]:
final_batch.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.01
    ),
    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)
history_final = final_batch.fit(train_dataset,validation_data=val_dataset,epochs=8,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 330s 3s/step - accuracy: 0.4128 - loss: 1.7776 - val_accuracy: 0.0686 - val_loss: 2.7875
Epoch 2/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 411s 4s/step - accuracy: 0.4680 - loss: 1.6507 - val_accuracy: 0.3395 - val_loss: 2.1322
Epoch 3/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 331s 3s/step - accuracy: 0.4749 - loss: 1.5764 - val_accuracy: 0.4794 - val_loss: 1.5081
Epoch 4/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 329s 3s/step - accuracy: 0.4872 - loss: 1.4914 - val_accuracy: 0.2856 - val_loss: 2.2148
Epoch 5/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 332s 3s/step - accuracy: 0.4822 - loss: 1.4350 - val_accuracy: 0.2989 - val_loss: 1.8700
Epoch 6/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 362s 3s/step - accuracy: 0.4304 - loss: 1.4423 - val_accuracy: 0.5260 - val_loss: 1.1750
Epoch 7/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 332s 3s/step - accuracy: 0.4926 - loss: 1.3505 - val_accuracy: 0.5340 - val_loss: 1.1912
Epoch 8/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 330s 3s/step - accuracy: 0.4959 - loss: 1.3190 - val_accuracy: 0.

In [43]:
train_loss, train_accuracy = final_batch.evaluate(train_dataset)
val_loss, val_accuracy = final_batch.evaluate(val_dataset)
test_loss, test_accuracy = final_batch.evaluate(test_dataset)


110/110 ━━━━━━━━━━━━━━━━━━━━ 69s 613ms/step - accuracy: 0.5337 - loss: 1.1458
24/24 ━━━━━━━━━━━━━━━━━━━━ 14s 558ms/step - accuracy: 0.5260 - loss: 1.1750
24/24 ━━━━━━━━━━━━━━━━━━━━ 14s 566ms/step - accuracy: 0.5176 - loss: 1.1948


In [44]:
cnn_batchnorm_64.save("models/final_batch.keras")